# Day 13：同一世界下的 Year 2 过程模块

这一讲演示 ChemWorld 如何在同一个 `WorldLaw` 下开放结晶、蒸馏、连续流和电化学任务。它们不是新的小游戏，而是同一个物理化学世界里的不同 task slice。

## 学习目标

- 查看 Year 2 过程模块是否进入统一 `WorldLawSpec`。
- 使用 scripted baseline 跑通四个新增 task。
- 汇总 final assay 中的过程指标。
- 用 validator 理解状态依赖的动作前置条件。

In [ ]:
from __future__ import annotations

import gymnasium as gym
import pandas as pd

import chemworld  # noqa: F401
from chemworld.agents.base import HistoryRecord
from chemworld.agents.event import ScriptedChemistryAgent
from chemworld.tasks import get_task
from chemworld.world import world_law_spec
from chemworld.wrappers import ActionMaskWrapper, validate_event_action


## 1. 查看统一世界规律

`WorldLawSpec` 记录所有模块。新增过程应出现在同一个 world law 里，而不是注册成独立环境。

In [ ]:
law = world_law_spec().to_dict()
modules = pd.DataFrame(law["ontology_registry"]["modules"])
modules[["module_id", "version"]]

## 2. 运行四个 Year 2 task

下面用同一个 scripted baseline 运行四个任务，并抓取每个任务最后一次 final assay。

In [ ]:
PROCESS_TASKS = [
    "reaction-to-crystallization",
    "reaction-to-distillation",
    "flow-reaction-optimization",
    "electrochemical-conversion",
]

def run_scripted_task(task_id: str, seed: int = 0) -> dict[str, float]:
    task = get_task(task_id)
    env = gym.make("ChemWorld", task_id=task_id, seed=seed)
    obs, info = env.reset(seed=seed)
    agent = ScriptedChemistryAgent()
    agent.reset(info, seed=seed)
    history: list[HistoryRecord] = []
    last_assay = None
    final_count = 0
    invalid_count = 0
    for _ in range(task.budget):
        action = agent.act(history)
        obs, reward, terminated, truncated, step_info = env.step(action)
        history.append(
            HistoryRecord(
                step=len(history) + 1,
                action=action,
                observation=obs,
                reward=reward,
                info=step_info,
            )
        )
        flags = step_info.get("constraint_flags", {})
        invalid_count += int(flags.get("precondition_failed", False))
        if action.get("operation") == "measure" and action.get("instrument") == "final_assay":
            final_count += 1
            last_assay = obs
        if terminated or truncated:
            break
    env.close()
    assay = last_assay or obs
    keys = [
        "score",
        "yield",
        "crystal_yield",
        "crystal_purity",
        "distillate_purity",
        "distillate_recovery",
        "flow_conversion",
        "electrochemical_selectivity",
        "energy_efficiency",
        "safety_risk",
        "cost",
    ]
    row = {"task_id": task_id, "steps": len(history), "final_assay_count": final_count, "invalid_count": invalid_count}
    for key in keys:
        value = assay.get(key)
        row[key] = float(value[0]) if value is not None and value[0] == value[0] else None
    return row

results = pd.DataFrame([run_scripted_task(task_id) for task_id in PROCESS_TASKS])
results

## 3. 对比过程指标

不同 task 的主指标不同，所以不能把所有任务混成一个总榜。这里先做可视化检查：每个任务是否产生了自己关心的过程指标。

In [ ]:
plot_cols = [
    "crystal_purity",
    "distillate_purity",
    "flow_conversion",
    "electrochemical_selectivity",
    "energy_efficiency",
]

def bar(value: float | None, width: int = 24) -> str:
    if value is None:
        return "未观测"
    filled = round(max(0.0, min(1.0, value)) * width)
    return "█" * filled + "░" * (width - filled) + f" {value:.3f}"

visual = results[["task_id", *plot_cols]].copy()
for col in plot_cols:
    visual[col] = visual[col].map(bar)
visual

## 4. 状态依赖前置条件

物理世界不能允许“没结晶就过滤”。`ActionMaskWrapper` 和 `validate_event_action` 会把这种错误提前暴露给学生或 LLM-agent。

In [ ]:
env = ActionMaskWrapper(gym.make("ChemWorld", task_id="reaction-to-crystallization", seed=0))
obs, info = env.reset(seed=0)

bad_filter = validate_event_action({"operation": "filter_crystals"}, env)
bad_filter["valid"], bad_filter["invalid_reasons"]

In [ ]:
for action in [
    {"operation": "add_solvent", "volume_L": 0.028, "solvent": 2},
    {"operation": "add_reagent", "amount_mol": 0.010},
    {"operation": "seed_crystals", "seed_mass_g": 0.006},
    {"operation": "cool_crystallize", "target_temperature_K": 278.15, "duration_s": 1200.0},
]:
    obs, reward, terminated, truncated, info = env.step(action)

good_filter = validate_event_action({"operation": "filter_crystals"}, env)
env.close()
good_filter["valid"], good_filter["invalid_reasons"]

## 小结

Year 2 的关键不是“多几个环境”，而是同一个物理化学世界逐步开放更多操作。agent 如果能学到反应、分离、结晶、蒸馏、流动和电化学之间共享的结构，它才是在学习局部 world model，而不是刷单个小游戏。